# Notebook 4: Knowledge Distillation

**What this does (in simple terms):**

Imagine a brilliant professor (teacher model) who really understands
a subject. Now we want to train a student who is smaller and faster
but still pretty good. Instead of learning only from textbooks (hard labels:
"this is real" or "this is fake"), the student also learns from the
professor's notes — the professor's confidence levels on each sample.

For example, the teacher might say: "I'm 92% sure this is fake and
8% sure it's real." That 8% contains useful information — maybe the
fake audio was very convincing. The student learns from these soft
predictions, becoming smarter than if it just learned from yes/no labels.

Steps:
1. Load the trained teacher from Notebook 3
2. Run all training audio through the teacher to get soft predictions
3. Build a student model (frozen Wav2Vec2 + lightweight AASIST-L)
4. Train the student using both hard labels and teacher's soft predictions

**Run on Colab Pro with GPU. Takes ~3-6 hours.**

## 4.1 Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install transformers -q

In [ ]:
import os, json, random, numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
import torchaudio
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from transformers import Wav2Vec2Model
from sklearn.metrics import accuracy_score, roc_curve
from tqdm import tqdm
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

In [ ]:
PREPROCESSED_DIR = Path("/content/preprocessed")
CHECKPOINT_DIR = Path("/content/drive/MyDrive/deepfake_project/checkpoints")

TARGET_SR = 16000; MAX_DURATION = 6; MAX_SAMPLES = TARGET_SR * MAX_DURATION
BATCH_SIZE = 8; NUM_EPOCHS = 10
LEARNING_RATE_BACKEND = 1e-4; WEIGHT_DECAY = 1e-4

# Knowledge distillation settings
KD_TEMPERATURE = 3.0   # Higher = softer probability distribution
KD_ALPHA = 0.5          # 0.5 = equal weight hard loss and soft loss

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

In [ ]:
# Verify data and teacher checkpoint exist
assert PREPROCESSED_DIR.exists(), "Preprocessed data not found! Run Notebook 1 first."
assert (CHECKPOINT_DIR / "teacher_best.pt").exists(), "Teacher checkpoint not found! Run Notebook 3 first."
print("Preprocessed data: OK")
print("Teacher checkpoint: OK")

## 4.2 Load Data

In [ ]:
def load_protocol(path):
    df = pd.read_csv(path, sep=" ", header=None,
                     names=["speaker_id", "audio_id", "system_id", "attack_type", "label"])
    df["label_int"] = (df["label"] == "spoof").astype(int)
    return df

PROTOCOL_DIR = PREPROCESSED_DIR / "protocols"
train_protocol = load_protocol(PROTOCOL_DIR / "ASVspoof2019.LA.cm.train.trn.txt")
dev_protocol = load_protocol(PROTOCOL_DIR / "ASVspoof2019.LA.cm.dev.trl.txt")
metadata = pd.read_csv(PREPROCESSED_DIR / "metadata.csv")
print(f"Train: {len(train_protocol)} | Dev: {len(dev_protocol)}")

In [ ]:
# --- Augmentation (same as Notebook 3) ---
def rawboost_convolutive(a, sr=TARGET_SR):
    n = len(a); s = np.fft.rfft(a); fr = np.fft.rfftfreq(n, d=1.0/sr)
    for _ in range(np.random.randint(1,5)):
        f=np.random.uniform(100,sr//2-100); bw=np.random.uniform(50,300)
        s *= 1.0 - np.exp(-0.5*((fr-f)/(bw/2+1e-8))**2)
    return np.fft.irfft(s, n=n).astype(np.float32)

def rawboost_impulsive(a):
    nl=np.random.uniform(0.001,0.01)
    return (a + nl*np.random.randn(len(a))*np.abs(a)).astype(np.float32)

def rawboost_augment(a):
    a=a.copy()
    if np.random.rand()<0.5: a=rawboost_convolutive(a)
    if np.random.rand()<0.5: a=rawboost_impulsive(a)
    return a

def codec_augment(a, sr=TARGET_SR):
    t=torch.from_numpy(a).unsqueeze(0).float()
    if np.random.rand()<0.5:
        t=torchaudio.functional.mu_law_decoding(torchaudio.functional.mu_law_encoding(t,256),256)
    else:
        t=torchaudio.transforms.Resample(sr,8000)(t); t=torchaudio.transforms.Resample(8000,sr)(t)
    return t.squeeze(0).numpy()

def apply_augmentation(audio_tensor):
    a=audio_tensor.numpy().copy()
    if np.random.rand()<0.7: a=rawboost_augment(a)
    if np.random.rand()<0.3: a=codec_augment(a)
    return torch.from_numpy(a).float()

In [ ]:
# --- Standard dataset (for generating soft predictions — no augmentation) ---
class ASVspoofDataset(Dataset):
    def __init__(self, protocol_df, audio_dir, metadata_df, is_train=False):
        self.protocol = protocol_df.reset_index(drop=True)
        self.audio_dir = Path(audio_dir)
        self.is_train = is_train
        self.length_lookup = {}
        for _, r in metadata_df.iterrows():
            if r.get("status") == "success":
                self.length_lookup[r["audio_id"]] = int(r["original_length"])
    def __len__(self): return len(self.protocol)
    def __getitem__(self, idx):
        row = self.protocol.iloc[idx]
        audio, _ = torchaudio.load(str(self.audio_dir / f"{row['audio_id']}.flac"))
        audio = audio.squeeze(0)
        if self.is_train: audio = apply_augmentation(audio)
        mx = torch.max(torch.abs(audio))
        if mx > 0: audio = audio / mx
        ol = self.length_lookup.get(row["audio_id"], MAX_SAMPLES)
        mask = torch.zeros(MAX_SAMPLES); mask[:ol] = 1.0
        return {"audio": audio, "mask": mask,
                "label": torch.tensor(row["label_int"], dtype=torch.long),
                "audio_id": row["audio_id"], "attack_type": row["attack_type"]}

## 4.3 Load Teacher Model & Generate Soft Predictions

In [ ]:
# --- Teacher architecture (must match Notebook 3 exactly) ---
class GraphAttentionLayer(nn.Module):
    def __init__(self, in_dim, out_dim, dropout=0.1):
        super().__init__()
        self.W = nn.Linear(in_dim, out_dim, bias=False)
        self.a_src = nn.Linear(out_dim, 1, bias=False)
        self.a_dst = nn.Linear(out_dim, 1, bias=False)
        self.leaky_relu = nn.LeakyReLU(0.2)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        h = self.W(x)
        attn = self.leaky_relu(self.a_src(h) + self.a_dst(h).transpose(-2,-1))
        return torch.bmm(self.dropout(F.softmax(attn, dim=-1)), h)

class MultiHeadGraphAttention(nn.Module):
    def __init__(self, in_dim, out_dim, n_heads=2, dropout=0.1):
        super().__init__()
        self.heads = nn.ModuleList([GraphAttentionLayer(in_dim, out_dim, dropout) for _ in range(n_heads)])
        self.norm = nn.LayerNorm(out_dim * n_heads)
    def forward(self, x):
        return self.norm(torch.cat([h(x) for h in self.heads], dim=-1))

class AASSISTBackend(nn.Module):
    def __init__(self, input_dim=1024, hidden_dim=160, n_heads=2, dropout=0.1):
        super().__init__()
        self.projection = nn.Sequential(nn.Linear(input_dim, hidden_dim), nn.GELU(),
                                         nn.LayerNorm(hidden_dim), nn.Dropout(dropout))
        self.spectral_gat = MultiHeadGraphAttention(hidden_dim, hidden_dim, n_heads, dropout)
        self.spectral_pool = nn.AdaptiveAvgPool1d(1)
        self.temporal_gat = MultiHeadGraphAttention(hidden_dim, hidden_dim, n_heads, dropout)
        self.temporal_pool = nn.AdaptiveAvgPool1d(1)
        cd = hidden_dim * n_heads * 2
        self.hetero_attention = nn.Sequential(nn.Linear(cd, hidden_dim), nn.GELU(),
                                               nn.Dropout(dropout), nn.Linear(hidden_dim, hidden_dim))
        self.classifier = nn.Sequential(nn.Linear(hidden_dim, 64), nn.GELU(),
                                         nn.Dropout(dropout), nn.Linear(64, 2))
    def forward(self, f):
        x = self.projection(f)
        sp = self.spectral_pool(self.spectral_gat(x).transpose(1,2)).squeeze(-1)
        tp = self.temporal_pool(self.temporal_gat(x).transpose(1,2)).squeeze(-1)
        return self.classifier(self.hetero_attention(torch.cat([sp, tp], dim=-1)))

class TeacherModel(nn.Module):
    def __init__(self, w2v_name="facebook/wav2vec2-xls-r-300m", hidden_dim=160):
        super().__init__()
        self.wav2vec2 = Wav2Vec2Model.from_pretrained(w2v_name)
        self.wav2vec2.feature_extractor._freeze_parameters()
        self.backend = AASSISTBackend(input_dim=self.wav2vec2.config.hidden_size, hidden_dim=hidden_dim)
    def forward(self, audio, mask=None):
        if mask is not None: mask = mask.long()
        return self.backend(self.wav2vec2(audio, attention_mask=mask).last_hidden_state)

In [ ]:
# Load teacher
print("Loading teacher model...")
teacher = TeacherModel().to(DEVICE)
ckpt = torch.load(str(CHECKPOINT_DIR / "teacher_best.pt"), map_location=DEVICE)
teacher.load_state_dict(ckpt["model_state_dict"])
teacher.eval()
print(f"Teacher loaded (epoch {ckpt['epoch']}, Dev EER: {ckpt['dev_eer']*100:.2f}%)")

In [ ]:
# Check if soft predictions already exist (skip regeneration)
SOFT_PRED_PATH = CHECKPOINT_DIR / "teacher_soft_predictions.pt"

if SOFT_PRED_PATH.exists():
    print("Soft predictions already exist. Loading...")
    sp_data = torch.load(str(SOFT_PRED_PATH), map_location="cpu")
    teacher_logits = sp_data["logits"]
    teacher_ids = sp_data["audio_ids"]
else:
    print("Generating soft predictions (this takes ~30-60 min)...")
    # Non-augmented dataset for consistent soft labels
    train_no_aug = ASVspoofDataset(train_protocol, PREPROCESSED_DIR/"train", metadata, is_train=False)
    train_no_aug_loader = DataLoader(train_no_aug, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

    teacher_logits, teacher_ids = [], []
    with torch.no_grad():
        for batch in tqdm(train_no_aug_loader, desc="Generating soft predictions"):
            audio = batch["audio"].to(DEVICE)
            mask = batch["mask"].to(DEVICE)
            logits = teacher(audio, mask)
            teacher_logits.append(logits.cpu())
            teacher_ids.extend(batch["audio_id"])

    teacher_logits = torch.cat(teacher_logits, dim=0)

    # Save to Google Drive
    torch.save({"logits": teacher_logits, "audio_ids": teacher_ids}, str(SOFT_PRED_PATH))
    print(f"Saved soft predictions: {teacher_logits.shape}")

print(f"Soft predictions ready: {teacher_logits.shape[0]} samples")

In [ ]:
# Visualize teacher confidence
probs = F.softmax(teacher_logits, dim=-1).numpy()
plt.figure(figsize=(10, 4))
plt.hist(probs[:, 1], bins=50, color="steelblue", edgecolor="white")
plt.xlabel("Teacher P(spoof)"); plt.ylabel("Count")
plt.title("Teacher Confidence Distribution")
plt.tight_layout(); plt.show()

## 4.4 Distillation Dataset

Same as regular dataset, but each sample also includes the
teacher's soft prediction for that audio file.

In [ ]:
class DistillationDataset(Dataset):
    def __init__(self, protocol_df, audio_dir, metadata_df, teacher_logits_dict, is_train=True):
        self.protocol = protocol_df.reset_index(drop=True)
        self.audio_dir = Path(audio_dir)
        self.teacher_logits_dict = teacher_logits_dict
        self.is_train = is_train
        self.length_lookup = {}
        for _, r in metadata_df.iterrows():
            if r.get("status") == "success":
                self.length_lookup[r["audio_id"]] = int(r["original_length"])

    def __len__(self): return len(self.protocol)

    def __getitem__(self, idx):
        row = self.protocol.iloc[idx]
        audio, _ = torchaudio.load(str(self.audio_dir / f"{row['audio_id']}.flac"))
        audio = audio.squeeze(0)
        if self.is_train: audio = apply_augmentation(audio)
        mx = torch.max(torch.abs(audio))
        if mx > 0: audio = audio / mx
        ol = self.length_lookup.get(row["audio_id"], MAX_SAMPLES)
        mask = torch.zeros(MAX_SAMPLES); mask[:ol] = 1.0
        t_logit = self.teacher_logits_dict.get(row["audio_id"], torch.zeros(2))
        return {"audio": audio, "mask": mask,
                "label": torch.tensor(row["label_int"], dtype=torch.long),
                "teacher_logits": t_logit,
                "audio_id": row["audio_id"], "attack_type": row["attack_type"]}

In [ ]:
teacher_logits_dict = {aid: teacher_logits[i] for i, aid in enumerate(teacher_ids)}

distill_dataset = DistillationDataset(
    train_protocol, PREPROCESSED_DIR/"train", metadata, teacher_logits_dict, is_train=True)
distill_loader = DataLoader(distill_dataset, batch_size=BATCH_SIZE, shuffle=True,
                             num_workers=2, pin_memory=True)

dev_dataset = ASVspoofDataset(dev_protocol, PREPROCESSED_DIR/"dev", metadata, is_train=False)
dev_loader = DataLoader(dev_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=2, pin_memory=True)

print(f"Distillation: {len(distill_loader)} batches | Dev: {len(dev_loader)} batches")

## 4.5 Student Model (Frozen Wav2Vec2 + AASIST-L)

In [ ]:
class AASSISTLBackend(nn.Module):
    """Lightweight AASIST (~85K params). Same idea, smaller dimensions."""
    def __init__(self, input_dim=1024, hidden_dim=64, n_heads=2, dropout=0.1):
        super().__init__()
        self.projection = nn.Sequential(nn.Linear(input_dim, hidden_dim), nn.GELU(),
                                         nn.LayerNorm(hidden_dim), nn.Dropout(dropout))
        self.spectral_gat = MultiHeadGraphAttention(hidden_dim, hidden_dim, n_heads, dropout)
        self.spectral_pool = nn.AdaptiveAvgPool1d(1)
        self.temporal_gat = MultiHeadGraphAttention(hidden_dim, hidden_dim, n_heads, dropout)
        self.temporal_pool = nn.AdaptiveAvgPool1d(1)
        cd = hidden_dim * n_heads * 2
        self.hetero_attention = nn.Sequential(nn.Linear(cd, hidden_dim), nn.GELU(),
                                               nn.Dropout(dropout), nn.Linear(hidden_dim, hidden_dim))
        self.classifier = nn.Sequential(nn.Linear(hidden_dim, 32), nn.GELU(),
                                         nn.Dropout(dropout), nn.Linear(32, 2))
    def forward(self, f):
        x = self.projection(f)
        sp = self.spectral_pool(self.spectral_gat(x).transpose(1,2)).squeeze(-1)
        tp = self.temporal_pool(self.temporal_gat(x).transpose(1,2)).squeeze(-1)
        return self.classifier(self.hetero_attention(torch.cat([sp, tp], dim=-1)))

class StudentModel(nn.Module):
    def __init__(self, w2v_name="facebook/wav2vec2-xls-r-300m", hidden_dim=64):
        super().__init__()
        self.wav2vec2 = Wav2Vec2Model.from_pretrained(w2v_name)
        for p in self.wav2vec2.parameters(): p.requires_grad = False  # Frozen!
        self.backend = AASSISTLBackend(
            input_dim=self.wav2vec2.config.hidden_size, hidden_dim=hidden_dim)

    def forward(self, audio, mask=None):
        if mask is not None: mask = mask.long()
        with torch.no_grad():
            features = self.wav2vec2(audio, attention_mask=mask).last_hidden_state
        return self.backend(features)

In [ ]:
def count_params(m, trainable=True):
    if trainable: return sum(p.numel() for p in m.parameters() if p.requires_grad)
    return sum(p.numel() for p in m.parameters())

student = StudentModel().to(DEVICE)
print(f"Student total params:     {count_params(student, False):,}")
print(f"Student trainable params: {count_params(student, True):,}")

## 4.6 Loss Functions & Training Loop

In [ ]:
class OCSoftmax(nn.Module):
    def __init__(self, m_real=0.9, m_fake=0.2, alpha=20.0):
        super().__init__()
        self.m_real = m_real; self.m_fake = m_fake; self.alpha = alpha
    def forward(self, logits, labels):
        rm = (labels == 0).float(); sm = (labels == 1).float()
        mod = logits.clone()
        mod[:, 0] -= self.m_real * rm; mod[:, 1] -= self.m_fake * sm
        return F.cross_entropy(self.alpha * mod, labels)

def kd_loss(student_logits, teacher_logits, temperature):
    s = F.log_softmax(student_logits / temperature, dim=-1)
    t = F.softmax(teacher_logits / temperature, dim=-1)
    return F.kl_div(s, t, reduction="batchmean") * (temperature ** 2)

def compute_eer(labels, scores):
    fpr, tpr, th = roc_curve(labels, scores, pos_label=1)
    fnr = 1 - tpr; idx = np.nanargmin(np.abs(fpr - fnr))
    return (fpr[idx] + fnr[idx]) / 2, th[idx]

@torch.no_grad()
def evaluate_student(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0; all_s, all_p, all_l = [], [], []
    for b in tqdm(dataloader, desc="Evaluating"):
        a = b["audio"].to(device); m = b["mask"].to(device); l = b["label"].to(device)
        logits = model(a, m); total_loss += criterion(logits, l).item()
        all_s.extend(F.softmax(logits, dim=-1)[:,1].cpu().numpy())
        all_p.extend(torch.argmax(logits, dim=-1).cpu().numpy())
        all_l.extend(l.cpu().numpy())
    la = np.array(all_l); sa = np.array(all_s)
    eer, _ = compute_eer(la, sa)
    return {"loss": total_loss/len(dataloader), "eer": eer,
            "accuracy": accuracy_score(la, np.array(all_p))}

In [ ]:
hard_criterion = OCSoftmax().to(DEVICE)
optimizer = torch.optim.AdamW(student.backend.parameters(), lr=LEARNING_RATE_BACKEND,
                               weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

# --- Check for existing student checkpoint ---
STUDENT_CKPT_PATH = CHECKPOINT_DIR / "student_latest.pt"
STUDENT_BEST_PATH = CHECKPOINT_DIR / "student_best.pt"

start_epoch = 1
best_student_eer = float("inf")
history = {"train_loss": [], "hard_loss": [], "kd_loss": [],
           "train_acc": [], "dev_eer": [], "dev_loss": []}

if STUDENT_CKPT_PATH.exists():
    print("Found student checkpoint! Resuming...")
    ckpt = torch.load(str(STUDENT_CKPT_PATH), map_location=DEVICE)
    student.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    scheduler.load_state_dict(ckpt["scheduler_state_dict"])
    start_epoch = ckpt["epoch"] + 1
    best_student_eer = ckpt["best_dev_eer"]
    history = ckpt["history"]
    print(f"Resuming from epoch {start_epoch} (best EER: {best_student_eer*100:.2f}%)")
else:
    print("No student checkpoint found. Starting fresh.")

In [ ]:
print("=" * 60)
print(f"STUDENT TRAINING (KD): Epochs {start_epoch} to {NUM_EPOCHS}")
print(f"  Temperature: {KD_TEMPERATURE} | Alpha: {KD_ALPHA}")
print("=" * 60)

for epoch in range(start_epoch, NUM_EPOCHS + 1):
    print(f"\n{'='*40} Epoch {epoch}/{NUM_EPOCHS} {'='*40}")

    student.train()
    ep_loss, ep_hard, ep_kd = 0, 0, 0
    ep_preds, ep_labels = [], []

    pbar = tqdm(distill_loader, desc=f"Epoch {epoch} [Student]")
    for batch in pbar:
        audio = batch["audio"].to(DEVICE); mask = batch["mask"].to(DEVICE)
        labels = batch["label"].to(DEVICE); t_logits = batch["teacher_logits"].to(DEVICE)

        optimizer.zero_grad()
        s_logits = student(audio, mask)
        h_loss = hard_criterion(s_logits, labels)
        k_loss = kd_loss(s_logits, t_logits, KD_TEMPERATURE)
        loss = KD_ALPHA * h_loss + (1 - KD_ALPHA) * k_loss

        loss.backward()
        torch.nn.utils.clip_grad_norm_(student.parameters(), max_norm=1.0)
        optimizer.step()

        ep_loss += loss.item(); ep_hard += h_loss.item(); ep_kd += k_loss.item()
        ep_preds.extend(torch.argmax(s_logits, dim=-1).cpu().numpy())
        ep_labels.extend(labels.cpu().numpy())
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    n = len(distill_loader)
    t_loss, t_hard, t_kd = ep_loss/n, ep_hard/n, ep_kd/n
    t_acc = accuracy_score(ep_labels, ep_preds)

    dev_res = evaluate_student(student, dev_loader, hard_criterion, DEVICE)
    scheduler.step()

    history["train_loss"].append(t_loss); history["hard_loss"].append(t_hard)
    history["kd_loss"].append(t_kd); history["train_acc"].append(t_acc)
    history["dev_eer"].append(dev_res["eer"]); history["dev_loss"].append(dev_res["loss"])

    print(f"  Loss: {t_loss:.4f} (Hard: {t_hard:.4f}, KD: {t_kd:.4f})")
    print(f"  Train Acc: {t_acc:.4f} | Dev EER: {dev_res['eer']*100:.2f}%")

    if dev_res["eer"] < best_student_eer:
        best_student_eer = dev_res["eer"]
        torch.save({"epoch": epoch, "model_state_dict": student.state_dict(),
                     "dev_eer": best_student_eer}, str(STUDENT_BEST_PATH))
        print(f"  ★ New best student! (EER: {best_student_eer*100:.2f}%)")

    torch.save({"epoch": epoch, "model_state_dict": student.state_dict(),
                 "optimizer_state_dict": optimizer.state_dict(),
                 "scheduler_state_dict": scheduler.state_dict(),
                 "best_dev_eer": best_student_eer, "history": history},
               str(STUDENT_CKPT_PATH))
    print(f"  Checkpoint saved (epoch {epoch})")

print(f"\n{'='*60}")
print(f"STUDENT TRAINING COMPLETE! Best Dev EER: {best_student_eer*100:.2f}%")
print(f"{'='*60}")

## 4.7 Training Plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(history["train_loss"], "o-", label="Total")
axes[0].plot(history["hard_loss"], "s-", alpha=0.7, label="Hard")
axes[0].plot(history["kd_loss"], "^-", alpha=0.7, label="KD")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
axes[0].set_title("Student Losses"); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(history["train_acc"], "o-", color="green")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy")
axes[1].set_title("Student Train Accuracy"); axes[1].grid(True, alpha=0.3)

axes[2].plot([e*100 for e in history["dev_eer"]], "o-", color="red")
axes[2].set_xlabel("Epoch"); axes[2].set_ylabel("EER (%)")
axes[2].set_title("Student Dev EER"); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(str(CHECKPOINT_DIR / "student_training_plots.png"), dpi=150, bbox_inches="tight")
plt.show()
print("\n✅ Notebook 4 complete. Proceed to Notebook 5: Evaluation.")